# Day 15 — Confidence Intervals

Today we'll learn how to estimate a population value using a range rather than only one number.

### Learning goals
- Point estimates and confidence intervals
- Interpret confidence levels
- Calculate z and t confidence intervals
- Visualize intervals and explore sample size

We use NumPy, pandas, Matplotlib, and SciPy. If SciPy is missing, run `pip install scipy` in your environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
np.random.seed(42)

## 1. Point estimate vs. confidence interval
A **point estimate** is one number used to estimate a population parameter. A **confidence interval (CI)** is a range produced by a method designed to capture the true parameter at a stated long-run rate.

A common form is:

\[\text{estimate} \pm \text{margin of error}\]

## 2. What does 95% confidence mean?
If we repeatedly collected random samples in the same way and built a 95% interval each time, about 95% of those intervals would contain the fixed population mean in the long run, assuming the method's conditions are met. It does **not** mean there is a 95% probability that the fixed mean lies inside this already-calculated frequentist interval.

## 3. Z interval when population SD is known
For a mean with known population standard deviation \(\sigma\):

\[\bar{x}\pm z^*\frac{\sigma}{\sqrt{n}}\]

For a two-sided 95% interval, \(z^*\approx1.96\).

In [ ]:
sample_mean = 50
population_sd = 12
n = 64
confidence = 0.95
alpha = 1 - confidence
z_star = stats.norm.ppf(1 - alpha/2)
se = population_sd / np.sqrt(n)
margin = z_star * se
print('Standard error:', round(se, 3))
print('Margin of error:', round(margin, 3))
print(f'95% z interval: ({sample_mean-margin:.2f}, {sample_mean+margin:.2f})')

## 4. T interval when population SD is unknown
Usually the population SD is unknown, so use the sample SD `s` and the t distribution:

\[\bar{x}\pm t^*\frac{s}{\sqrt{n}}\]

For a one-sample mean interval, degrees of freedom are \(n-1\). The t distribution accounts for extra uncertainty from estimating the population SD.

In [ ]:
measurements = np.array([48, 52, 47, 51, 54, 49, 50, 53, 46, 52, 51, 49])
n = len(measurements)
mean = np.mean(measurements)
s = np.std(measurements, ddof=1)
confidence = 0.95
alpha = 1 - confidence
t_star = stats.t.ppf(1-alpha/2, df=n-1)
se = s / np.sqrt(n)
margin = t_star * se
low, high = mean-margin, mean+margin
print('n:', n, '| mean:', round(mean, 2), '| sample SD:', round(s, 2))
print('Estimated SE:', round(se, 2))
print(f'95% t interval: ({low:.2f}, {high:.2f})')

## 5. Reusable function for a mean confidence interval
This function uses a t interval. It assumes independent observations and is most reliable when the data are reasonably well-behaved. Strong skewness or extreme outliers can be a problem for very small samples.

In [ ]:
def mean_confidence_interval(data, confidence=0.95):
    data = np.asarray(data, dtype=float)
    n = len(data)
    if n < 2:
        raise ValueError('Provide at least two observations.')
    mean = np.mean(data)
    sd = np.std(data, ddof=1)
    se = sd / np.sqrt(n)
    alpha = 1 - confidence
    critical = stats.t.ppf(1 - alpha/2, df=n-1)
    margin = critical * se
    return mean, mean-margin, mean+margin

example = [21, 19, 22, 20, 24, 18, 23, 21]
for level in [0.90, 0.95, 0.99]:
    mean, low, high = mean_confidence_interval(example, level)
    print(f'{level:.0%} CI: mean={mean:.2f}, interval=({low:.2f}, {high:.2f}), width={high-low:.2f}')

Higher confidence levels usually create wider intervals because the method is designed to capture the true value more often across repeated samples.

## 6. Visualize a confidence interval

In [ ]:
data = np.array([48, 52, 47, 51, 54, 49, 50, 53, 46, 52, 51, 49])
mean, low, high = mean_confidence_interval(data)
plt.figure(figsize=(8, 2.5))
plt.errorbar(mean, 1, xerr=[[mean-low], [high-mean]], fmt='o', capsize=8)
plt.axvline(mean, linestyle='--', label=f'Sample mean = {mean:.2f}')
plt.yticks([]); plt.xlabel('Measurement')
plt.title(f'95% CI: ({low:.2f}, {high:.2f})'); plt.legend(); plt.show()

## 7. What changes interval width?
- **Larger sample:** usually narrower interval.
- **More variable data:** wider interval.
- **Higher confidence level:** wider interval.
- **Lower confidence level:** narrower interval.

In [ ]:
population = np.random.normal(loc=100, scale=15, size=100_000)
rows = []
for n in [10, 25, 50, 100, 400]:
    sample = np.random.choice(population, size=n, replace=False)
    mean, low, high = mean_confidence_interval(sample)
    rows.append({'sample_size': n, 'sample_mean': mean, 'lower': low, 'upper': high, 'width': high-low})
results = pd.DataFrame(rows)
display(results.round(2))

In [ ]:
plt.figure(figsize=(8, 4))
for i, row in results.iterrows():
    plt.plot([row['lower'], row['upper']], [i, i], marker='|', linewidth=2)
    plt.plot(row['sample_mean'], i, 'o')
plt.yticks(range(len(results)), [f"n={n}" for n in results['sample_size']])
plt.xlabel('Estimated population mean'); plt.title('95% t intervals for different sample sizes')
plt.grid(axis='x', alpha=0.3); plt.show()

## 8. Common mistakes
1. A 95% CI does not mean 95% of individual data values lie in the interval.
2. When population SD is unknown, a t interval commonly uses sample SD.
3. Some intervals miss the true value; the confidence level is a long-run property.
4. Biased or dependent samples can make standard intervals misleading.
5. A narrow interval does not remove bias.

## 9. Practice exercises
1. Explain point estimate vs. confidence interval.
2. A sample has mean 80, sample SD 10, and n=25. Find estimated SE.
3. Why is a 99% interval usually wider than a 90% interval?
4. Run the function at 90%, 95%, and 99%. Compare widths.
5. Why can a large biased sample still mislead?

**Answer hints:** (1) One number vs. a range with a stated long-run coverage. (2) 10/√25 = **2**. (3) Higher confidence needs a larger critical value. (4) Width usually increases with confidence. (5) Formulas do not automatically correct selection or measurement bias.

## Day 15 summary
- Confidence intervals estimate a population parameter with a range.
- The interval is estimate ± margin of error.
- Use a z interval when population SD is known under suitable conditions.
- Use a t interval for a mean when population SD is unknown and sample SD is used.
- Larger samples often narrow intervals; higher confidence widens them.

**Next: Day 16 — Hypothesis Testing Basics:** null and alternative hypotheses, test statistics, p-values, significance levels, and interpretation.